In [ ]:
// ==== Setup ====

import java.util.*;

# `%` with negative numbers

## Circular array, going backward

- Forward: `(i + 1) % capacity`
- Backward: `(i - 1) % capacity`? At `i = 0`?

In [ ]:
System.out.println(13 % 8);
System.out.println(-1 % 8);
System.out.println(-9 % 8);
System.out.println(-8 % 8);

## Remainder vs. modulo

- Java `/` on `int`s rounds toward `0`: `-9 / 8` is `-1`.
- `%` is the remainder of that division: `a == (a / b) * b + a % b`.
- So `a % b` has the **sign of `a`**: `-9 % 8` is `-1`.
- Math: `a mod b` is always in `0..b-1` (for `b > 0`): `-9 mod 8` is `7`.
- `Math.floorMod(a, b)`: rounds `a / b` **down** (`Math.floorDiv`): same as math `mod`.

## Adding the modulus

- If `a >= -b`, then `a + b >= 0`: `(a + b) % b` is `a mod b`.
- Backward in a circular array: `(i - 1 + capacity) % capacity`.

In [ ]:
System.out.println((-1 % 8) + " " + Math.floorMod(-1, 8));
System.out.println((-9 % 8) + " " + Math.floorMod(-9, 8));
System.out.println((13 % 8) + " " + Math.floorMod(13, 8));
System.out.println((-9 / 8) + " " + Math.floorDiv(-9, 8));
System.out.println((-1 + 8) % 8 + " " + (-9 + 8) % 8);   // correct only for a >= -b

- `boolean isOdd(int x) { return x % 2 == 1; }`: correct?
- Same values for `a >= 0`: when is `%` enough?

# Deques

## Deque ADT

- **Double-ended queue**: add and remove at **both** ends.
- `addFirst`, `addLast`, `removeFirst`, `removeLast`, `peekFirst`, `peekLast`
- Stack: only one end. Queue: add at one end, remove at the other.

In [ ]:
interface MyDeque<E> {
    void addFirst(E e);
    void addLast(E e);

    // throw NoSuchElementException if the deque is empty
    E removeFirst();
    E removeLast();
    E peekFirst();
    E peekLast();

    boolean isEmpty();
    int size();
}

## Linked deque

- `MyDLinkedListDouble`: all six operations `O(1)` already.
- Singly linked, with `head` and `tail`: which operation is not `O(1)`?

## Circular array deque

`ArrayQueue` does `addLast` and `removeFirst`. The two new operations move the other way:
- `addFirst`: `front` moves one cell **back**: `Math.floorMod(front - 1, capacity)`
- `removeLast`: the last element is at `(front + size - 1) % capacity`

In [ ]:
class MyArrayDeque<E> implements MyDeque<E> {
    private E[] data;
    private int front;
    private int size;

    @SuppressWarnings("unchecked")
    MyArrayDeque() { data = (E[]) new Object[4]; }

    public int size() { return size; }
    public boolean isEmpty() { return size == 0; }

    // index in data of element i, counting from the front
    private int index(int i) { return (front + i) % data.length; }

    @SuppressWarnings("unchecked")
    private void grow() {
        E[] bigger = (E[]) new Object[2 * data.length];
        for (int i = 0; i < size; i++)
            bigger[i] = data[index(i)];
        data = bigger;
        front = 0;
    }

    private void checkNotEmpty() {
        if (size == 0) throw new NoSuchElementException("empty deque");
    }

    public void addFirst(E e) {
        if (size == data.length) grow();
        front = Math.floorMod(front - 1, data.length);
        data[front] = e;
        size++;
    }

    public void addLast(E e) {
        if (size == data.length) grow();
        data[index(size)] = e;
        size++;
    }

    public E removeFirst() {
        checkNotEmpty();
        E e = data[front];
        data[front] = null;
        front = index(1);
        size--;
        return e;
    }

    public E removeLast() {
        checkNotEmpty();
        int last = index(size - 1);
        E e = data[last];
        data[last] = null;
        size--;
        return e;
    }

    public E peekFirst() { checkNotEmpty(); return data[front]; }
    public E peekLast()  { checkNotEmpty(); return data[index(size - 1)]; }

    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder("[");
        for (int i = 0; i < size; i++) {
            sb.append(data[index(i)]);
            if (i < size - 1) sb.append(", ");
        }
        return sb.append("]").toString();
    }

    public String internals() {
        return Arrays.toString(data) + "  front=" + front + "  size=" + size;
    }
}

In [ ]:
MyArrayDeque<String> dq = new MyArrayDeque<>();
dq.addLast("c");
dq.addFirst("b");               // front wraps to the last cell
dq.addFirst("a");
System.out.println(dq + "   " + dq.internals());

dq.addLast("d");
dq.addLast("e");                // full: grows
System.out.println(dq + "   " + dq.internals());

System.out.println(dq.removeLast() + " " + dq.removeFirst() + " " + dq + "   " + dq.internals());

- `removeFirst` uses `index(1)`, `addFirst` uses `floorMod`. Why not `floorMod` in both?
- Cost of each operation?

# Lists, stacks, queues in `java.util`

## Interfaces

```
Iterable<E>
 └─ Collection<E>
     ├─ List<E>
     └─ Queue<E>
         └─ Deque<E>
```

- `Collection<E>`: what lists and queues have in common: `add(e)`, `remove(o)`, `contains(o)`, `size()`, `isEmpty()`, `iterator()`, ...
- `List<E>`: access by position, `get(i)`, `add(i, e)`, ...
- `Queue<E>`, `Deque<E>`: access at the ends only.
- No interface for stacks.

## Classes

| class | structure | implements |
|---|---|---|
| `ArrayList<E>` | resizable array | `List<E>` |
| `LinkedList<E>` | doubly linked list | `List<E>`, `Deque<E>` |
| `ArrayDeque<E>` | resizable circular array | `Deque<E>` |
| `Vector<E>` | resizable array (older `ArrayList`) | `List<E>` |
| `Stack<E>` | `extends Vector<E>` | `List<E>` |

In [ ]:
List<Collection<Integer>> all = List.of(new ArrayList<>(), new LinkedList<>(), new ArrayDeque<>(), new Stack<>());
System.out.printf("%-12s %6s %6s %6s%n", "", "List", "Queue", "Deque");
for (Collection<Integer> c : all)
    System.out.printf("%-12s %6b %6b %6b%n", c.getClass().getSimpleName(),
                      c instanceof List, c instanceof Queue, c instanceof Deque);

## Ours and theirs

| ADT | ours | `java.util` |
|---|---|---|
| list | `MyListDouble`: `MyArrayListDouble`, `MyLinkedListDouble`, `MyDLinkedListDouble` | `List<E>`: `ArrayList`, `LinkedList` |
| stack | `MyStack<E>`: `ArrayStack`, `LinkedStack` | no interface: `Deque<E>` |
| queue | `MyQueue<E>`: `ArrayQueue`, `LinkedQueue` | `Queue<E>`: `ArrayDeque`, `LinkedList` |
| deque | `MyDeque<E>`: `MyArrayDeque` | `Deque<E>`: `ArrayDeque`, `LinkedList` |

- `ArrayList`: as `MyArrayListDouble`
- `LinkedList`: as `MyDLinkedListDouble`, plus the `Deque` methods
- `ArrayDeque`: as `MyArrayDeque`

## Costs

| | `ArrayList` | `LinkedList` | `ArrayDeque` |
|---|---|---|---|
| `get(i)`, `set(i, e)` | `O(1)` | `O(N)` | not available |
| add, remove at the front | `O(N)` | `O(1)` | `O(1)` amortized |
| add, remove at the back | `O(1)` amortized | `O(1)` | `O(1)` amortized |
| add, remove in the middle | `O(N)` | `O(N)` | not available |
| remove through an iterator | `O(N)` | `O(1)` | `O(N)` |

- A circular array could give `get(i)` in `O(1)`. `ArrayDeque` does not offer it: why might that be?
- Java 21: `List` and `Deque` share `addFirst`, `addLast`, `getFirst`, `getLast`, `removeFirst`, `removeLast`, `reversed()` (interface `SequencedCollection<E>`). Cost of `addFirst` on an `ArrayList`?

## `LinkedList`: one class, two interfaces

- The **reference** type decides which methods can be called.
- The **object** decides how they run.

In [ ]:
LinkedList<String> names = new LinkedList<>(List.of("b", "c"));
List<String> asList = names;        // same object, three references
Deque<String> asDeque = names;

asDeque.addFirst("a");
asList.add(1, "a2");
System.out.println(names + " " + asList.get(1) + " " + asDeque.peekLast());

- `asDeque.get(1)`: does it compile? `asList.addFirst("z")`?
- `Deque<String> d = new LinkedList<>()` or `new ArrayDeque<>()`: which one, and why?

## `Stack<E>` is a class

- `push`, `pop`, `peek`, `empty()`
- It is a `Vector`: everything a `List` does, too.

In [ ]:
Stack<String> js = new Stack<>();
js.push("bottom");
js.push("middle");
js.push("top");
System.out.println(js + " " + js.peek());

js.add(0, "below the bottom");      // inherited from Vector
System.out.println(js + " " + js.get(1));

- Is this still a stack?
- Documentation of `Stack`: "A more complete and consistent set of LIFO stack operations is provided by the `Deque` interface and its implementations, which should be used in preference to this class."
- Bottom line: pretend this class does not exist

## `Queue<E>`

Each operation in two versions:

| | throws an exception | returns a special value |
|---|---|---|
| add at the back | `add(e)` | `offer(e)`: `false` if full |
| remove the front | `remove()` | `poll()`: `null` if empty |
| front element | `element()` | `peek()`: `null` if empty |

- Ours: `enqueue`, `dequeue`, `peek`, all throwing.
- When can a queue be full?

## `Deque<E>`

- `interface Deque<E> extends Queue<E>`
- Same two versions, at each end:

| | throws an exception | returns a special value |
|---|---|---|
| add | `addFirst(e)`, `addLast(e)` | `offerFirst(e)`, `offerLast(e)` |
| remove | `removeFirst()`, `removeLast()` | `pollFirst()`, `pollLast()` |
| examine | `getFirst()`, `getLast()` | `peekFirst()`, `peekLast()` |

- Queue methods: add at the **last**, remove at the **first**.
- Stack methods: `push`, `pop`, `peek`, all at the **first**.

In [ ]:
Deque<Integer> stack = new ArrayDeque<>();
stack.push(1);
stack.push(2);
stack.push(3);
System.out.println(stack + " " + stack.pop() + " " + stack.peek());

Queue<Integer> queue = new ArrayDeque<>();
queue.offer(1);
queue.offer(2);
queue.offer(3);
System.out.println(queue + " " + queue.poll() + " " + queue.peek());

queue.clear();
System.out.println(queue.poll());

- `stack` prints as `[3, 2, 1]`: why?
- `ArrayDeque` does not accept `null` elements: `offer(null)` throws. Why?
- `Queue<Integer> q = new ArrayDeque<>()`: can we call `q.push(4)`?
- `ArrayDeque` vs. `LinkedList` as a `Deque`: which one, and why?

## Exercises

### 1. Balanced delimiters
Do the `( )`, `[ ]`, `{ }` in a string match?

| string | balanced? |
|---|---|
| `{[()]}` | yes |
| `a[i] = f(b[j]);` | yes |
| `([)]` | no |
| `(()` | no |
| `())` | no |

- A closing delimiter must match the **most recent** opening one that is still unmatched.
- `boolean balanced(String s)`, with a `Deque<Character>`.
- `([)]`, `(()`, `())`: three different ways to fail. Which line of your code catches each?
- `pop()` or `poll()`: which one makes the empty case simpler?
- Cost? Largest possible size of the stack?
- Only `(` and `)`: is a stack needed? Three counters, one per type of delimiter: enough?

In [ ]:
boolean balanced(String s) {
    Deque<Character> open = new ArrayDeque<>();
    for (char c : s.toCharArray()) {
        if (c == '(' || c == '[' || c == '{') {
            open.addFirst(c);
        }
        else if (c == ')') {
            if (open.isEmpty()) return false;
            if (open.removeFirst() != '(') return false;
        }
        else if (c == ']') {
            if (open.isEmpty()) return false;
            if (open.removeFirst() != '[') return false;
        }
        else if (c == '}') {
            if (open.isEmpty()) return false;
            if (open.removeFirst() != '{') return false;
        }
    }
    if (open.isEmpty()) return true;
    return false;
}

In [ ]:
// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

In [ ]:
check(balanced("{[()]}") && balanced("a[i] = f(b[j]);") && balanced(""), "balanced");
check(!balanced("([)]") && !balanced("(()") && !balanced("())"), "not balanced");

`int firstError(String s)`: `-1` if `s` is balanced, otherwise the position of the first character that makes it unbalanced. If opening delimiters are left unclosed, the position of the first one. What should the stack hold?

In [ ]:
int firstError(String s) {
    for (int i = 0; i < s.length(); i++) {
        char c = s.charAt(i);

    }
    return -1;
}

In [ ]:
check(firstError("{[()]}") == -1, "balanced");
check(firstError("([)]") == 2, "wrong closing");
check(firstError("())") == 2, "closing with nothing open");
check(firstError("x(y[z]") == 1, "never closed");
check(firstError("(()") == 0, "never closed, the outer one");

### 2. Reversing a queue
`<E> void reverse(Queue<E> q)`: reverse the order of the elements in `q`, using one `Deque<E>` as a stack.
- Cost?
- Without the `Deque`, with recursion: where is the stack now?

In [ ]:
<E> void reverse(Queue<E> q) {
    Deque<E> stack = new ArrayDeque<>();
    
}

In [ ]:
Queue<Integer> r = new ArrayDeque<>();
for (int i = 1; i <= 5; i++) r.offer(i);
reverse(r);
check(r.toString().equals("[5, 4, 3, 2, 1]"), "reversed: " + r);

### 3. Stacks from queues, queues from stacks
A queue from two stacks `in`, `out`:
- `enqueue`: push on `in`.
- `dequeue`: when is it necessary to move elements from `in` to `out`?
- Cost of a single `dequeue`, worst case? Total cost of `N` `enqueue` and `N` `dequeue`?

A stack from two queues:
- Which operation is `O(N)`? Can both `push` and `pop` be `O(1)`? Can it be done with a single queue?

### 4. Choosing
Which class, and through which interface?
- The history of the "back" button of a browser.
- Jobs waiting for a printer.
- A playlist: append songs, play song number `i`.
- A playlist where songs are often inserted right after the current one, while iterating.
- The last 100 lines of a log: add at one end, drop from the other, sometimes read line `i`.